### Foundations of Finance
# Python Practice #2: Bond Basics

---
## 1. **Run:** Prepare the environment

Let's start by importing (and installing when needed) all the necessary libraries for the task:
- [math](https://docs.python.org/3/library/math.html): built-in math stuff.
- [numpy](https://numpy.org/): numerical methods.
- [numpy-financial](https://numpy.org/numpy-financial/): for the IRR.
- [pandas](https://pandas.pydata.org/): data manipulation.
- [plotly](https://plotly.com/): interactive plots.  

All required packages must be installed before they are imported. The installation procedure depends on your specific Python setup (e.g., Anaconda). Using the default package manager `pip` should work in most cases.  

**IMPORTANT:** Are you using Google Colab? Then you probably need to install `numpy-financial` from the command-line.  

The command-line is available in jupyter notebooks using `!` at the beginning of the line (you have to remove the comment `#` for the command below to work).

In [ ]:
#obligatie -> ik koop een obligatie en ik krijg couponbetalingen + de face value 
#aan het einde terug -> de face value is wat jouw beloofd wordt betaald te worden
#bij vervaldatum

#zero coupon -> je hebt geen couponbetaling
!pip install numpy-financial

Import required packages

In [ ]:
# basic math stuff
import math

# Pandas
import pandas as pd

# NumPy
import numpy as np


# Graphical library
import plotly.express as px
import plotly.graph_objs as go

---
## 2. **Exercise:** Prepare a table with all the required bond data

To solve problems with bonds, we typically build a table (a pandas `DataFrame`) to store all relevant information.  
For example, for a semi-annual bond (i.e., two coupons per year), 3-year maturity, face value 100, and a 10% coupon rate, the desired table looks like the one below:

| period | tenor | cashflow  | yield | present_value | ... |         
|--------|-------|-----------|-------|---------------|-----|
| 0      | 0     | 0         | ...   | ...           | ... |
| 1      | 0.5   | 5         | ...   | ...           | ... |
| 2      | 1.0   | 5         | ...   | ...           | ... |
| 3      | 1.5   | 5         | ...   | ...           | ... |
| 4      | 2.0   | 5         | ...   | ...           | ... |
| 5      | 2.5   | 5         | ...   | ...           | ... |
| 6      | 3.0   | 105       | ...   | ...           | ... |

Write a function that returns a `DataFrame` like the one above (just `period`, `tenor`, and `cashflow` columns), given the following input arguments (about 9 lines of code):
- `face_value` of the bond.
- `maturity`in years.
- `coupons_per_year`
- `coupon_rate`

The resulting `DataFrame` will contain a sequential index as well, that's fine, ignore it.

**IMPORTANT:** if `coupon_rate` and `coupons_per_year` are equal to zero, then it is *Zero Coupon Bond* (ZCB). **Your code must be able to handle that!**

In [ ]:
#face_value: De nominale waarde van de obligatie.
#maturity: De looptijd van de obligatie in jaren.
#coupons_per_year: Het aantal couponbetalingen per jaar.
#coupon_rate: De jaarlijkse couponrente als decimaal getal.

**TEST YOUR CODE:** the following code should return the table above (up to the `cashflow` column). Naming explicitly the arguments is optional in this case, but it improves readability.

In [ ]:
import pandas as pd

def bond_data(face_value, maturity, coupons_per_year, coupon_rate):
    # Calculate the total number of periods
    periods = int(maturity * coupons_per_year) if coupons_per_year != 0 else maturity
    
    #als er geen coupons zijn= zero coupon is je periode gewoon gelijk aan je maturity
    
    
    # Calculate the tenor for each period
    if coupons_per_year == 0:  # For Zero Coupon Bonds #je maakt lijst met i is maturity zet je op i plek
        tenor = [i for i in range(maturity + 1)] #+1 want de laatste wordt niet meegenomen
        #bij zero coupon = tenor gelijk aan periode -> i for i is gewoon hetzelfde 
    else:
        tenor = [i / coupons_per_year for i in range(periods + 1)]
        
        #bijv voor i=periode 1 -> dit deel je door 2(coupons per year)
        #deze voeg je later toe als variabele in de dataframe
    
    # Calculate the cashflow for each period
    if coupons_per_year == 0:  # Zero Coupon Bond
        cashflow = [0] * (maturity + 1)#gehele maturity is je cashflow nul tot de laatste betaling
        cashflow[-1] = face_value #de laatste cashflow is je face value -> dit is bedrag wat aan je is beloofd 
    else:  # Regular bond
        coupon_payment = face_value * (coupon_rate / coupons_per_year)
        cashflow = [0] + [coupon_payment] * periods #je moet startwaarde hebben op plek 0 
        cashflow[-1] += face_value  # Add face value to the last cashflow
        #bij regular is je laatste cashflow -> face value + cashflow berekend op die plek
    
    # Create the DataFrame-> doe dit door een lijst te maken van een variabele 
    df = pd.DataFrame({
        'period': list(range(len(tenor))), #je wilt hier weer lengte bepalen -> je begint bij plek 0 -> dus je hoeft niet +1 te doen
        'tenor': tenor,
        'cashflow': cashflow
    })
    
    return df

# Test the function with a regular bond
TEST_BOND = bond_data(
    face_value=100,
    maturity=3,
    coupons_per_year=2,
    coupon_rate=0.1)
print("Regular Bond:")
print(TEST_BOND)

# Test the function with a Zero Coupon Bond
TEST_ZCB = bond_data(
    face_value=100,
    maturity=3,
    coupons_per_year=0,
    coupon_rate=0.0)
print("\nZero Coupon Bond:")
print(TEST_ZCB)


TEST_BOND = bond_data(
    face_value=100,
    maturity=3,
    coupons_per_year=2,
    coupon_rate=0.1)

TEST_BOND

**TEST YOUR CODE:** Let's turn this bond into a ZCB by setting `coupons_per_year = 0`. Can you explain the differences?

| period | tenor | cashflow  |        
|--------|-------|-----------|
| 0      | 0     | 0         |
| 1      | 1     | 0         |
| 2      | 2     | 0         |
| 3      | 3     | 100       |

In [ ]:
TEST_ZCB = bond_data(
    face_value=100,
    maturity=3,
    coupons_per_year=0,
    coupon_rate=0.0)

TEST_ZCB

---
## 3. **Exercise:** Price a bond

Given the `DataFrame` from the previous exercise and a `discount_date`, how would you calculate the price of the bond? An NPV perhaps?

The bond price is the sum of the present values of the payments and the face value.

$$
P = \frac{C}{(1+r)^1} + \frac{C}{(1+r)^2} + \frac{C}{(1+r)^3} + \cdots + \frac{FV + C}{(1+r)^n}
$$

For semi-annual:
$$
P = \frac{C}{\left( 1+\frac{r}{2} \right)^1} + \frac{C}{\left(1+\frac{r}{2}\right)^2} + \frac{C}{\left(1+\frac{r}{2}\right)^3} + \cdots + \frac{FV + C}{\left(1+\frac{r}{2}\right)^{2n}}
$$


Write a function that calculates the price (i.e., NPV) of the bond's stream of cashflows (column `cashflow`) at a given `yield_to_maturity` (about 5 lines of code).  
Don't forget to take into account the `coupons_per_year` for the APR!

Let the function return two items:
- A new table, containing additional columns for `yield` and `present_value`.
- The price.

**IMPORTANT:** You just did something very similar in the previous practice!

**TEST YOUR CODE:** the following code should return 100. Do you know why?

In [ ]:
price, bond = bond_value(
    TEST_BOND,
    yield_to_maturity = 0.1,
    coupons_per_year=2)

print(price)
bond

**TEST YOUR CODE:** the following code should return 75.13.

In [ ]:
import pandas as pd

#yield to maturity -> verwachte rendement op obligatie 
#present value 


def bond_value(data, yield_to_maturity, coupons_per_year):
    # Handle Zero Coupon Bonds
    if coupons_per_year == 0:
        data['yield'] = yield_to_maturity   #dit is de data die in je functie staat
        data['present_value'] = data['cashflow'] / (1 + yield_to_maturity)**data['period']
    else:
        data['yield'] = yield_to_maturity
        data['present_value'] = data['cashflow'] / (1 + yield_to_maturity / coupons_per_year)**(data['period'])
    
    # Calculate the price of the bond as the sum of present values
    price = data['present_value'].sum()
    #The bond price is the sum of the present values of the payments and the face value.
    #vgm zit in die cashflow al de facevalue dus je hoeft dat niet meer erbij te doen
    
    return price, data

# Assuming TEST_BOND and TEST_ZCB are DataFrames created using the bond_data function
price, bond = bond_value(
    TEST_BOND,                #je hebt hier de eerdere dataframe die je erin stop
    yield_to_maturity=0.1,
    coupons_per_year=2
)
print("Price for TEST_BOND:", price)
print(bond)


price, bond = bond_value(
    TEST_ZCB,
    yield_to_maturity = 0.1,
    coupons_per_year=0)

#Hier wordt de bond_value-functie aangeroepen met de gegeven parameters en de 
#geretourneerde waarden worden toegewezen aan de variabelen price en bond. price 
#zal de berekende prijs van de obligatie bevatten, terwijl bond mogelijk aanvullende 
#informatie over de obligatie bevat

print("the price is", price)


---
## 4. **Exercise:** Calculate the Yield to Maturity (YTM)

The **Yield to Maturity (YTM)** is the discount rate required to have a bond price equals to **B**.

$$
B = \sum_{t=0}^{n}\frac{CF_t}{(1+YTM)^t}
$$

Write a function that calculates the YTM (i.e., IRR) of the bond's stream of cashflows (column `cashflow`), taking into account the `coupons_per_year` for the APR (about 4 lines of code).

**IMPORTANT:** Don't forget to "overwrite" the first cashflow with the given bond price (negative), then calculate everything, and finally leave it as zero again (as it was)!

In [ ]:
#In deze ytm_func-functie wordt de geschatte YTM (ytm_guess) gebruikt om de totale 
#waarde van de kasstromen te berekenen. De variabele total wordt geïnitialiseerd op nul
#en vervolgens wordt elke kasstroom, gewogen naar de huidige waarde, bij deze totale 
#waarde opgeteld. De functie retourneert het verschil tussen de
#berekende totale waarde van de kasstromen en de gegeven prijs van de obligatie (price).

#De code for t, cashflow in enumerate(data['cashflow']): is een for-lus die iteratief 
#door de elementen van de 'cashflow'-kolom in de DataFrame data gaat. 
#De enumerate-functie wordt gebruikt om zowel de waarde van de kasstroom als de 
#bijbehorende index (tijdstap) tijdens de iteratie te verkrijgen.

def bond_yield_to_maturity(data,
                           price,
                           coupons_per_year):
    def ytm_func(ytm_guess):
        total = 0
        for t, cashflow in enumerate(data['cashflow']):
            total += cashflow / (1 + ytm_guess / coupons_per_year) ** t
        return total - price
    
    
    
    


**TEST YOUR CODE:** the following line should return 10%.  

In [ ]:
from scipy.optimize import root_scalar

def bond_yield_to_maturity(data, price, coupons_per_year):
    # Overwrite the first cashflow with the given bond price (negative)
    original_first_cashflow = data.loc[0, 'cashflow']#de eerste in de rij van kolom cashflow
    data.loc[0, 'cashflow'] = -price #-1000 in de lijst net als h1 
    
    # Define the function to find the root for
    def npv_func(ytm):                 #je doet zip() als je twee kolommen in een functie wil gebruiken uit dataframe
        if coupons_per_year == 0:  # For Zero Coupon Bonds-. de cashflows (allemaal 0 )+ cashflow bij uitbetaling 
            return sum(cf / (1 + ytm)**t for cf, t in zip(data['cashflow'], data['period']))
        else:  # For regular bonds
            return sum(cf / (1 + ytm / coupons_per_year)**(t * coupons_per_year) for cf, t in zip(data['cashflow'], data['period']))
    #net als bij econ eng-> als niet zcp -> dan rente/quatarly(4) -> t * 4
    
    
    # Use root_scalar to find the YTM
    result = root_scalar(npv_func, bracket=[0.001, 0.2], method='brentq')
    ytm = result.root
    #blijkbaar moet je om ytm te vinden functie npv 
    # Restore the original first cashflow
    data.loc[0, 'cashflow'] = original_first_cashflow
    
    return ytm, data

# Assuming TEST_BOND and TEST_ZCB are DataFrames created using the bond_data function
irr, data = bond_yield_to_maturity(
    TEST_BOND,
    price=100,
    coupons_per_year=2
)
print("YTM for TEST_BOND:", irr)
print(data)

irr, data = bond_yield_to_maturity(
    TEST_ZCB,
    price=75.13,
    coupons_per_year=0
)
print("\nYTM for TEST_ZCB:", irr)
print(data)

irr, data = bond_yield_to_maturity(
    TEST_BOND,
    price=100,
    coupons_per_year=2)

print(irr)
data

**TEST YOUR CODE:** the following line should return 10%.

In [ ]:
irr, data = bond_yield_to_maturity(
    TEST_ZCB,
    price=75.13,
    coupons_per_year=0)

print(irr)
data

---
# 5. **Run:** Plot cashflows and present values

Assuming your code is right, this should work like a charm...

In [ ]:
def plot_cashflows_present_values(title, data):
    """ plot stream of cashflows and present values """
    
    # create base figure
    fig = go.Figure()

    # plot cashflows
    fig.add_trace(
        go.Bar(name='Cashflows', x=data['tenor'], y=data['cashflow']))

    # plot present values
    fig.add_trace(
        go.Bar(name='Present values', x=data['tenor'], y=data['present_value']))

    # title, axis labels and use percentages in axis
    fig.update_layout(
        #width=800,
        #height=600,
        yaxis= { 'tickformat': '.1f'},
        title=title,
        xaxis_title='tenor [years]',
        yaxis_title='money [$]',
    )

    fig.show()

Let's display the cashflows and present values for the `TEST_BOND`.

In [ ]:
plot_cashflows_present_values('Test bond', TEST_BOND)

And for the `TEST_ZCB` (a zero coupon bond).

In [ ]:
plot_cashflows_present_values('Test ZCB', TEST_ZCB)

---
# The End